# Vaidya Nidaan — Alzheimer's MRI classifier (v2)

Same architecture as `final_alzheimer_model.ipynb` (VGG-19 + dense head, 128×128 axial slices from OASIS-1), retrained with:

| | v1 | v2 |
|---|---|---|
| Data | `vedjosh/alzheimer-mri` — 38,430 slices (non-demented undersampled to balance) | `ninadaithal/imagesoasis` — all 86,437 slices, class weights instead of discarding data |
| Split | random split over slices | **split by patient** (OAS1_XXXX) — train / val / test share no brains |
| Input scaling | raw 0–255 RGB | VGG-19 ImageNet `preprocess_input` (what the backbone was trained on) |
| Output layer | `Dense(4)` on 2 classes | `Dense(2)` (or 3 for the CDR 0 / 0.5 / ≥1 variant) |
| Training | frozen backbone | head warm-up, then fine-tune `block5` with augmentation |
| Metrics | slice accuracy | slice **and** patient-level accuracy, balanced accuracy, sensitivity, specificity, ROC-AUC |

**Run on a GPU** (RunPod / Colab / Kaggle). Upload `kaggle.json` first (kaggle.com → Settings → API → Create New Token).
Outputs: `alzheimer_vgg19_v2.keras`, `metrics.json`, `split_subjects.json` → serve with `MODEL_PREPROCESS=vgg19`.

In [ ]:
# !pip install -q "tensorflow[and-cuda]==2.19.0" "keras==3.14.1" kagglehub scikit-learn pandas matplotlib seaborn
import os, re, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
tf.keras.utils.set_random_seed(SEED)
IMG = 128
TASK = "binary"          # "binary" (CDR 0 vs >= 0.5)  or  "3class" (CDR 0 / 0.5 / >= 1)
BATCH = 64
WARMUP_EPOCHS = 4        # head only, frozen backbone
FINETUNE_EPOCHS = 20     # block5 unfrozen (early stopping)
LIMIT = int(os.environ.get("VN_LIMIT", 0))   # >0 = quick smoke test on a subsample
OUT = os.environ.get("VN_OUT", "outputs")
os.makedirs(OUT, exist_ok=True)

## 1. Data — OASIS-1 axial slices

In [ ]:
import kagglehub

DATA = os.environ.get("VN_DATA") or kagglehub.dataset_download("ninadaithal/imagesoasis")
print("Dataset path:", DATA)

In [ ]:
# Folder name -> CDR group. imagesOASIS has 4 severity folders; the vedjosh subset has 2.
FOLDER_CDR = {"non demented": 0.0, "very mild dementia": 0.5, "mild dementia": 1.0,
              "moderate dementia": 2.0, "demented": None}
norm = lambda s: re.sub(r"[\s_\-]+", " ", s.strip().lower())

rows = []
for root, _, files in os.walk(DATA):
    cdr_key = norm(os.path.basename(root))
    if cdr_key not in FOLDER_CDR:
        continue
    for f in files:
        m = re.match(r"(OAS1_\d+)", f)
        if m and f.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append({"path": os.path.join(root, f), "subject": m.group(1), "folder": cdr_key, "cdr": FOLDER_CDR[cdr_key]})
df = pd.DataFrame(rows).sort_values("path").reset_index(drop=True)
if LIMIT:
    df = df.iloc[np.linspace(0, len(df) - 1, LIMIT).astype(int)].reset_index(drop=True)

if TASK == "binary":
    CLASSES = ["Non Demented", "Demented"]
    df["label"] = (df["folder"] != "non demented").astype(int)
else:
    assert df["cdr"].notna().all(), "3class needs the severity folders (imagesOASIS)"
    CLASSES = ["Non Demented (CDR 0)", "Very mild (CDR 0.5)", "Mild+ (CDR >= 1)"]
    df["label"] = df["cdr"].map(lambda c: 0 if c == 0 else (1 if c == 0.5 else 2))
K = len(CLASSES)

# Each patient carries one diagnosis.
assert (df.groupby("subject")["label"].nunique() == 1).all()
counts = df.groupby("label").agg(slices=("path", "size"), patients=("subject", "nunique"))
counts.index = [CLASSES[i] for i in counts.index]
print(counts)
print("\nSlices per original folder:\n", df["folder"].value_counts())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts["slices"].plot.bar(ax=ax[0], color="#18a886", title="Slices per class")
counts["patients"].plot.bar(ax=ax[1], color="#0b1437", title="Patients per class")
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

plt.figure(figsize=(14, 7))
for n, i in enumerate(random.Random(SEED).sample(range(len(df)), 18)):
    plt.subplot(3, 6, n + 1)
    plt.imshow(Image.open(df.path[i]).convert("RGB").resize((IMG, IMG)))
    plt.title(CLASSES[df.label[i]], fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()

## 2. Patient-level split

Every slice of a patient goes to exactly one of train / validation / test (70 / 15 / 15), stratified by diagnosis.
The test set therefore measures performance on **brains the model has never seen**.

In [ ]:
rng = np.random.default_rng(SEED)
subj_label = df.groupby("subject")["label"].first()
split = {}
for y in range(K):
    subs = np.array(sorted(subj_label[subj_label == y].index))
    rng.shuffle(subs)
    n_test, n_val = max(1, round(0.15 * len(subs))), max(1, round(0.15 * len(subs)))
    for s in subs[:n_test]: split[s] = "test"
    for s in subs[n_test:n_test + n_val]: split[s] = "val"
    for s in subs[n_test + n_val:]: split[s] = "train"
df["split"] = df["subject"].map(split)

sets = {k: set(df.subject[df.split == k]) for k in ("train", "val", "test")}
assert not (sets["train"] & sets["test"]) and not (sets["train"] & sets["val"]) and not (sets["val"] & sets["test"])
print(df.groupby(["split", "label"]).agg(slices=("path", "size"), patients=("subject", "nunique")))

## 3. Load images and build input pipelines

In [ ]:
def load(paths):
    # Same loading as v1: PIL resize to 128x128, RGB, 0-255 (kept as uint8 to save memory).
    X = np.zeros((len(paths), IMG, IMG, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        X[i] = np.asarray(Image.open(p).convert("RGB").resize((IMG, IMG)))
    return X

data = {k: (load(df.path[df.split == k].tolist()), df.label[df.split == k].to_numpy()) for k in ("train", "val", "test")}
subjects_test = df.subject[df.split == "test"].to_numpy()
for k, (X, y) in data.items():
    print(k, X.shape, np.bincount(y, minlength=K))

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.1),
])

def pipeline(X, y, train):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if train:
        ds = ds.shuffle(20000, seed=SEED)
    ds = ds.batch(BATCH).map(lambda x, t: (tf.cast(x, tf.float32), t), num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(lambda x, t: (augment(x, training=True), t), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.map(lambda x, t: (preprocess_input(x), t)).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = (pipeline(*data[k], train=(k == "train")) for k in ("train", "val", "test"))

# Class weights keep the rarer class from being ignored (instead of throwing data away).
y_train = data["train"][1]
class_weight = {c: len(y_train) / (K * max(1, (y_train == c).sum())) for c in range(K)}
print("class weights:", {CLASSES[c]: round(w, 3) for c, w in class_weight.items()})

## 4. Model — VGG-19 (ImageNet) + dense head

In [ ]:
base = VGG19(weights="imagenet", include_top=False, input_shape=(IMG, IMG, 3))
base.trainable = False
x = layers.Flatten(name="flatten")(base.output)
x = layers.Dense(256, activation="relu", name="dense")(x)
x = layers.Dropout(0.5, name="dropout")(x)
x = layers.Dense(128, activation="relu", name="dense_1")(x)
x = layers.Dropout(0.5, name="dropout_1")(x)
out = layers.Dense(K, activation="softmax", name="dense_2")(x)
model = tf.keras.Model(base.input, out, name="vaidya_nidaan_vgg19_v2")
model.summary(show_trainable=True)

## 5. Train — head warm-up, then fine-tune block5

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_warmup = model.fit(train_ds, validation_data=val_ds, epochs=WARMUP_EPOCHS, class_weight=class_weight)

base.trainable = True
for layer in base.layers:
    layer.trainable = layer.name.startswith("block5")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]
hist_ft = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, class_weight=class_weight, callbacks=callbacks)

In [ ]:
h = {k: hist_warmup.history[k] + hist_ft.history[k] for k in ("loss", "val_loss", "accuracy", "val_accuracy")}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame({"loss": h["loss"], "val_loss": h["val_loss"]}).plot(ax=ax[0], title="Loss")
pd.DataFrame({"accuracy": h["accuracy"], "val_accuracy": h["val_accuracy"]}).plot(ax=ax[1], title="Accuracy")
for a in ax: a.axvline(WARMUP_EPOCHS - 0.5, ls="--", c="grey")
plt.tight_layout(); plt.show()

## 6. Evaluate on held-out patients

In [ ]:
X_test, y_test = data["test"]
probs = model.predict(test_ds, verbose=0)
pred = probs.argmax(1)

def summarize(y, p, pr):
    cm = confusion_matrix(y, pr, labels=list(range(K)))
    recall = cm.diagonal() / cm.sum(1).clip(min=1)
    out = {"n": int(len(y)), "accuracy": float((pr == y).mean()), "balanced_accuracy": float(recall.mean()),
           "recall": {CLASSES[c]: float(recall[c]) for c in range(K)}, "confusion": cm.tolist()}
    if K == 2:
        out.update(sensitivity=float(recall[1]), specificity=float(recall[0]), auc=float(roc_auc_score(y, p[:, 1])))
    else:
        out["macro_auc_ovr"] = float(roc_auc_score(y, p, multi_class="ovr"))
    return out

slice_metrics = summarize(y_test, probs, pred)
print("SLICE LEVEL\n", classification_report(y_test, pred, target_names=CLASSES, digits=3))

# Patient level: average the slice probabilities of each brain -> one decision per patient.
pdf = pd.DataFrame(probs, columns=range(K)).assign(subject=subjects_test, y=y_test)
per_patient = pdf.groupby("subject").mean(numeric_only=True)
pp, py = per_patient[list(range(K))].to_numpy(), per_patient["y"].astype(int).to_numpy()
patient_metrics = summarize(py, pp, pp.argmax(1))
print("PATIENT LEVEL\n", classification_report(py, pp.argmax(1), target_names=CLASSES, digits=3))
print(json.dumps({k: v for k, v in patient_metrics.items() if k != "confusion"}, indent=1))

In [ ]:
fig, ax = plt.subplots(1, 3 if K == 2 else 2, figsize=(16 if K == 2 else 11, 4.5))
for a, (name, m) in zip(ax, [("Slices", slice_metrics), ("Patients", patient_metrics)]):
    sns.heatmap(np.array(m["confusion"]), annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=a, cbar=False)
    a.set_title(f"{name} — bal. acc {m['balanced_accuracy']:.3f}"); a.set_xlabel("Predicted"); a.set_ylabel("True")
if K == 2:
    for name, y, s in [("slices", y_test, probs[:, 1]), ("patients", py, pp[:, 1])]:
        fpr, tpr, _ = roc_curve(y, s)
        ax[2].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y, s):.3f})")
    ax[2].plot([0, 1], [0, 1], "--", c="grey"); ax[2].set_title("ROC — held-out patients"); ax[2].legend()
plt.tight_layout(); plt.show()

## 7. Grad-CAM++ on held-out patients

Same implementation as the app (`ml_service/inference.py`): gradients of the class log-odds with respect to `block5_conv4`.
Check whether the attribution sits on brain tissue (ventricles, medial temporal lobes) rather than the skull or orbits.

In [ ]:
grad_model = tf.keras.Model(model.inputs, [model.get_layer("block5_conv4").output, model.layers[-1].input])
W, b = model.layers[-1].get_weights()

def gradcam_pp(img_uint8):
    x = preprocess_input(img_uint8[None].astype("float32"))
    with tf.GradientTape() as tape:
        conv, pen = grad_model(x)
        logits = tf.matmul(pen, W) + b
        c = int(tf.argmax(logits[0]))
        others = tf.concat([logits[:, :c], logits[:, c + 1:]], axis=1)
        score = logits[:, c] - tf.reduce_logsumexp(others, axis=1)   # log-odds of the predicted class
    g = tape.gradient(score, conv)[0]; A = conv[0]
    g2, g3 = g * g, g * g * g
    denom = 2 * g2 + g3 * tf.reduce_sum(A, axis=(0, 1))
    alpha = g2 / tf.where(denom != 0, denom, tf.ones_like(denom))
    alpha /= tf.reduce_sum(alpha, axis=(0, 1)) + 1e-8
    w = tf.reduce_sum(alpha * tf.nn.relu(g), axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(w * A, -1)); cam /= tf.reduce_max(cam) + 1e-8
    return tf.image.resize(cam[..., None], (IMG, IMG)).numpy().squeeze(), c

idx = np.random.default_rng(SEED).choice(len(X_test), 8, replace=False)
plt.figure(figsize=(16, 4.5))
for n, i in enumerate(idx):
    cam, c = gradcam_pp(X_test[i])
    plt.subplot(2, 8, n + 1); plt.imshow(X_test[i]); plt.axis("off"); plt.title(f"true: {CLASSES[y_test[i]]}", fontsize=8)
    plt.subplot(2, 8, n + 9); plt.imshow(X_test[i]); plt.imshow(cam, cmap="jet", alpha=0.45); plt.axis("off"); plt.title(f"pred: {CLASSES[c]}", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Reference — the same backbone with a random *slice* split

Trains only the head on frozen features with v1's protocol (random split over slices), to show how much a
slice-level split inflates accuracy compared with the patient-level test above.

In [ ]:
RUN_SLICE_SPLIT_REFERENCE = True
if RUN_SLICE_SPLIT_REFERENCE:
    feat_model = VGG19(weights="imagenet", include_top=False, input_shape=(IMG, IMG, 3))
    X_all = np.concatenate([data[k][0] for k in ("train", "val", "test")])
    y_all = np.concatenate([data[k][1] for k in ("train", "val", "test")])
    feats = feat_model.predict(preprocess_input(X_all.astype("float32")), batch_size=256, verbose=0)
    perm = np.random.default_rng(SEED).permutation(len(y_all))
    te, tr = perm[: len(perm) // 5], perm[len(perm) // 5:]
    inp = layers.Input(feats.shape[1:])
    h_ = layers.Dropout(0.5)(layers.Dense(128, activation="relu")(layers.Dropout(0.5)(layers.Dense(256, activation="relu")(layers.Flatten()(inp)))))
    head = tf.keras.Model(inp, layers.Dense(K, activation="softmax")(h_))
    head.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    head.fit(feats[tr], y_all[tr], validation_split=0.2, epochs=30, batch_size=128, verbose=0,
             callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    slice_split_acc = float((head.predict(feats[te], verbose=0).argmax(1) == y_all[te]).mean())
    print(f"Random slice split (v1 protocol): accuracy {slice_split_acc:.3f}")
    print(f"Patient-level split (this model):  accuracy {patient_metrics['accuracy']:.3f} per patient, "
          f"{slice_metrics['accuracy']:.3f} per slice")

## 9. Save

In [ ]:
model.save(os.path.join(OUT, "alzheimer_vgg19_v2.keras"))
metrics = {"task": TASK, "classes": CLASSES, "dataset": DATA, "preprocess": "vgg19",
           "counts": counts.to_dict(), "slice": slice_metrics, "patient": patient_metrics,
           "slice_split_reference_accuracy": globals().get("slice_split_acc")}
json.dump(metrics, open(os.path.join(OUT, "metrics.json"), "w"), indent=2, default=float)
json.dump({k: sorted(v) for k, v in sets.items()}, open(os.path.join(OUT, "split_subjects.json"), "w"), indent=1)
print("Saved to", os.path.abspath(OUT), os.listdir(OUT))